# DP Optimizations

A **dynamic programming** recurrence is correct the moment you write it down — but its naive evaluation is often O(n²) or O(n³) because every state scans every earlier state. This notebook is about the second act: when the transition has *exploitable structure*, you can replace that inner scan with a smarter data structure or a smarter iteration order and drop a whole factor of `n`.

Four classic accelerators, each tied to a structural property of the recurrence:

- a **sliding-window minimum** → a monotonic deque (O(nk) → O(n)),
- a **linear-in-x transition** → the Convex Hull Trick (O(n²) → O(n log n) or O(n)),
- a **monotone optimal split** → divide-and-conquer DP (O(kn²) → O(kn log n)),
- a **quadrangle-inequality interval cost** → Knuth's optimization (O(n³) → O(n²)).

This builds directly on **dynamic programming** (you should be comfortable with states, recurrences, and tabulation first) and reuses the discard-what-can't-win idea from **monotonic stack & queue**. The heavier interval techniques here are the gateway to **advanced dynamic programming**. Every optimized routine below is asserted *equal to the naive DP it accelerates* over seeded-random inputs — the speedup is never allowed to change the answer.

**Contents**

1. **When the recurrence has structure** — the toolbox
2. **Monotonic-deque / sliding-window DP** — O(nk) → O(n)
3. **Convex Hull Trick** — O(n²) → O(n log n) / O(n)
4. **Divide-and-conquer DP** — O(kn²) → O(kn log n)
5. **Knuth's optimization** — O(n³) → O(n²)
6. **Which optimization fits which transition** + cheat-sheet

## 1. When the recurrence has structure — the toolbox

The naive bottom-up DP computes each state by looping over its predecessors:

```
for i in range(n):
    dp[i] = min(dp[j] + cost(j, i) for j in range(i))   # O(n) per state -> O(n^2) total
```

That inner `min` is doing real, repeated work that a structural property can collapse. The trick is always the same shape: **identify what the inner scan is recomputing, and amortize it.** Which tool applies depends entirely on the *form of the transition*:

| If the transition is… | the structure is… | use | drop |
|---|---|---|---|
| `min` over a **fixed-width window** of earlier `dp[j]` | a sliding window | **monotonic deque** | O(nk) → O(n) |
| `min_j ( m_j·x_i + b_j )` — **linear in a query value** `x_i` | a lower envelope of lines | **Convex Hull Trick** | O(n²) → O(n log n)/O(n) |
| `dp[g][i] = min_j dp[g-1][j] + C(j,i)`, optimal `j` **monotone in `i`** | monotone argmin | **divide-and-conquer** | O(kn²) → O(kn log n) |
| `dp[i][j] = min_k dp[i][k]+dp[k+1][j] + W(i,j)`, `W` obeys the **quadrangle inequality** | monotone split bounds | **Knuth's optimization** | O(n³) → O(n²) |

A unifying mental model: in all four, the set of candidate predecessors changes *slowly and monotonically* as `i` advances. The deque, the hull, the recursion bounds, and the `opt[][]` table are four different ways of carrying that candidate set forward instead of rebuilding it from scratch.

Everywhere below, `dpo_naive_*` is the obvious O(n²)/O(n³) DP, and the optimized version is checked against it on random inputs.

In [1]:
import random
from collections import deque
import time

random.seed(20260602)   # reproducible inputs for every fact-check in this notebook

def stress(naive, fast, gen, trials=400):
    """Assert two DPs agree on `trials` seeded-random inputs; return that they matched."""
    for _ in range(trials):
        args = gen()
        assert naive(*args) == fast(*args), (naive.__name__, args)
    return trials

print("seeded RNG ready; stress() will assert fast == naive on random inputs")

seeded RNG ready; stress() will assert fast == naive on random inputs


## 2. Monotonic-deque / sliding-window DP — O(nk) → O(n)

The simplest accelerator. Take a recurrence whose transition is a **minimum over a fixed-width window** of earlier states:

> `dp[i] = cost[i] + min( dp[j] for j in [i-k, i-1] )`,  with `dp[0] = cost[0]`.

(Story: the cheapest way to reach cell `i`, where from any cell you may step forward up to `k` positions, paying `cost[i]` to land on `i`.) The naive DP rescans up to `k` predecessors per state — **O(nk)**.

The structural insight is exactly the one from **monotonic stack & queue**: *an earlier state is useless the moment a later, smaller one enters the same window.* Keep a deque of candidate indices whose `dp` values are increasing; the front is always the window minimum. Each index is pushed and popped once → **O(n)**.

In [2]:
def dpo_sw_naive(cost, k):
    n = len(cost)
    INF = float("inf")
    dp = [INF] * n
    dp[0] = cost[0]
    for i in range(1, n):
        lo = max(0, i - k)
        dp[i] = cost[i] + min(dp[j] for j in range(lo, i))   # O(k) scan per state
    return dp[-1]

def dpo_sw_deque(cost, k):
    n = len(cost)
    dp = [0] * n
    dp[0] = cost[0]
    dq = deque([0])                       # indices; their dp values stay increasing
    for i in range(1, n):
        while dq and dq[0] < i - k:       # front slid out of the window
            dq.popleft()
        dp[i] = cost[i] + dp[dq[0]]       # window-min is always at the front
        while dq and dp[dq[-1]] >= dp[i]:
            dq.pop()                      # dp[i] dominates these -> they can never win
        dq.append(i)
    return dp[-1]

cost = [3, -2, 5, -1, 4, -3, 2, 7, -5, 1]
print("naive:", dpo_sw_naive(cost, 3))
print("deque:", dpo_sw_deque(cost, 3))

naive: -7
deque: -7


**Proof it's the same DP.** Identical answers across 400 random `(cost, k)` pairs — then a timing run at `n = 4000`, `k = 200` to *see* the O(nk) → O(n) drop.

In [3]:
def gen_sw():
    n = random.randint(1, 60)
    k = random.randint(1, 8)
    cost = [random.randint(-9, 9) for _ in range(n)]
    return (cost, k)

t = stress(dpo_sw_naive, dpo_sw_deque, gen_sw)
print(f"asserted equal on {t} random inputs")

big = [random.randint(-9, 9) for _ in range(4000)]
s = time.perf_counter(); a = dpo_sw_naive(big, 200);  t_naive = time.perf_counter() - s
s = time.perf_counter(); b = dpo_sw_deque(big, 200);  t_fast  = time.perf_counter() - s
print(f"n=4000 k=200:  naive {t_naive*1e3:6.1f} ms   deque {t_fast*1e3:5.2f} ms"
      f"   speedup x{t_naive/t_fast:.0f}   (match={a == b})")

asserted equal on 400 random inputs
n=4000 k=200:  naive   24.3 ms   deque  0.54 ms   speedup x45   (match=True)


## 3. Convex Hull Trick — O(n²) → O(n log n) / O(n)

Now the window isn't fixed — every earlier state is a candidate — but the transition is **linear in a query value** `x_i`:

> `dp[i] = base[i] + min_{j < i} ( slope[j] · x_i + dp[j] )`.

Read each predecessor `j` as a **line** `y = slope[j]·x + dp[j]`. Evaluating the transition at state `i` is asking for the **lowest line at `x = x_i`** — a query against the *lower envelope* of all lines added so far. Storing every line and scanning is O(n) per state → O(n²).

The structure: the lower envelope is a convex piecewise-linear function, so only the lines on its hull matter. Maintain that hull. In the friendly **monotonic** case — lines inserted in **non-increasing slope** order and queried at **non-decreasing** `x` — both insertion and query amortize to O(1) via a moving pointer, giving **O(n)** total. (General slopes/queries: keep the hull sorted and binary-search → O(n log n).)

In [4]:
def dpo_cht_naive(slope, x, base):
    n = len(x)
    dp = [0] * n
    dp[0] = base[0]
    for i in range(1, n):
        dp[i] = base[i] + min(slope[j] * x[i] + dp[j] for j in range(i))   # scan every line
    return dp

class LowerHull:
    """Lower envelope of lines y = m*x + c for minimization.
    Add lines with non-increasing slope; query with non-decreasing x -> O(1) amortized."""
    __slots__ = ("ms", "cs", "ptr")
    def __init__(self):
        self.ms = []; self.cs = []; self.ptr = 0

    def _redundant(self, i, j, k):
        # is the middle line j unnecessary? (cross-multiplied to stay integer-exact)
        return ((self.cs[k] - self.cs[i]) * (self.ms[i] - self.ms[j])
                <= (self.cs[j] - self.cs[i]) * (self.ms[i] - self.ms[k]))

    def add(self, m, c):
        if self.ms and self.ms[-1] == m:        # same slope -> keep only the lower intercept
            if self.cs[-1] <= c:
                return
            self.ms.pop(); self.cs.pop()
        self.ms.append(m); self.cs.append(c)
        while len(self.ms) >= 3 and self._redundant(-3, -2, -1):
            self.ms.pop(-2); self.cs.pop(-2)
        if self.ptr >= len(self.ms):
            self.ptr = len(self.ms) - 1

    def query(self, xq):
        if self.ptr >= len(self.ms):
            self.ptr = len(self.ms) - 1
        # walk the pointer forward while the next line is lower at xq
        while (self.ptr + 1 < len(self.ms)
               and self.ms[self.ptr + 1] * xq + self.cs[self.ptr + 1]
                   <= self.ms[self.ptr] * xq + self.cs[self.ptr]):
            self.ptr += 1
        return self.ms[self.ptr] * xq + self.cs[self.ptr]

def dpo_cht_fast(slope, x, base):
    n = len(x)
    dp = [0] * n
    dp[0] = base[0]
    hull = LowerHull()
    hull.add(slope[0], dp[0])
    for i in range(1, n):
        dp[i] = base[i] + hull.query(x[i])
        hull.add(slope[i], dp[i])
    return dp

# demo: queries x increasing, slopes decreasing (the monotonic-slopes variant)
x     = [0, 1, 3, 6, 10, 15]
slope = [9, 5, 4, 0, -3, -7]
base  = [0, 2, 1, 4, 0, 3]
print("naive:", dpo_cht_naive(slope, x, base))
print("hull :", dpo_cht_fast(slope, x, base))

naive: [0, 11, 27, 45, 45, 3]
hull : [0, 11, 27, 45, 45, 3]


**Proof it's the same DP.** Generate random instances respecting the monotonic-slopes contract (slopes sorted descending, query points sorted ascending) — including **negative** slopes and intercepts to exercise the integer-exact `_redundant` test — and assert the full `dp` array matches the O(n²) scan.

In [5]:
def gen_cht():
    n = random.randint(2, 50)
    x     = sorted(random.randint(0, 80) for _ in range(n))            # queries non-decreasing
    slope = sorted((random.randint(-40, 40) for _ in range(n)),        # slopes non-increasing
                   reverse=True)
    base  = [random.randint(0, 10) for _ in range(n)]
    return (slope, x, base)

t = stress(dpo_cht_naive, dpo_cht_fast, gen_cht)
print(f"asserted full dp[] equal on {t} random inputs (incl. negative slopes/intercepts)")

# timing: the O(n) hull vs the O(n^2) scan
N = 3000
x  = sorted(random.randint(0, 10**6) for _ in range(N))
sl = sorted((random.randint(-500, 500) for _ in range(N)), reverse=True)
bs = [random.randint(0, 50) for _ in range(N)]
s = time.perf_counter(); A = dpo_cht_naive(sl, x, bs); t_naive = time.perf_counter() - s
s = time.perf_counter(); B = dpo_cht_fast(sl, x, bs);  t_fast  = time.perf_counter() - s
print(f"n={N}:  naive {t_naive*1e3:7.1f} ms   hull {t_fast*1e3:5.2f} ms"
      f"   speedup x{t_naive/t_fast:.0f}   (match={A == B})")

asserted full dp[] equal on 400 random inputs (incl. negative slopes/intercepts)


n=3000:  naive   230.6 ms   hull  1.02 ms   speedup x225   (match=True)


## 4. Divide-and-conquer DP — O(kn²) → O(kn log n)

A *layered* recurrence: partition `n` items into `k` contiguous groups, minimizing total group cost.

> `dp[g][i] = min_{j < i} ( dp[g-1][j] + C(j+1, i) )`  —  group `g`'s right end is `i`, the previous group ended at `j`.

Per layer that's an O(n²) scan, so **O(kn²)** overall. The exploitable structure is **monotonicity of the optimal split**: let `opt[i]` be the best `j` for state `i`. If the cost `C` is well-behaved (a sufficient condition is the quadrangle inequality), then

> `opt[i] ≤ opt[i+1]`  — the best cut only moves rightward as `i` grows.

That lets a single layer be filled by **divide and conquer**: solve the midpoint `mid`, learn `opt[mid]`, then recurse knowing the left half's optima lie in `[…, opt[mid]]` and the right half's in `[opt[mid], …]`. Each level touches O(n) candidate slots over log n levels → **O(n log n)** per layer, **O(kn log n)** total.

Cost used below: `C(l, r) = (sum of a[l..r])²` — convex in the range sum, so it satisfies the monotone-`opt` condition.

In [6]:
def _prefix(a):
    pre = [0] * (len(a) + 1)
    for i, v in enumerate(a):
        pre[i + 1] = pre[i] + v
    return pre

def dpo_dc_naive(a, K):
    n = len(a)
    pre = _prefix(a)
    def C(l, r):                          # cost of grouping items l..r inclusive
        s = pre[r + 1] - pre[l]
        return s * s
    INF = float("inf")
    dp = [C(0, i) for i in range(n)]      # layer g=1: one group covering 0..i
    for _g in range(2, K + 1):
        cur = [INF] * n
        for i in range(n):
            for j in range(i):            # previous group ended at j -> new group j+1..i
                cur[i] = min(cur[i], dp[j] + C(j + 1, i))
        dp = cur
    return dp[n - 1]

def dpo_dc_fast(a, K):
    n = len(a)
    pre = _prefix(a)
    def C(l, r):
        s = pre[r + 1] - pre[l]
        return s * s
    INF = float("inf")
    dp = [C(0, i) for i in range(n)]
    for g in range(2, K + 1):
        cur = [INF] * n
        # fill cur[lo..hi]; the optimal split j is known to lie in [optlo, opthi]
        def solve(lo, hi, optlo, opthi):
            if lo > hi:
                return
            mid = (lo + hi) // 2
            best, bestj = INF, optlo
            for j in range(optlo, min(mid - 1, opthi) + 1):
                v = dp[j] + C(j + 1, mid)
                if v < best:
                    best, bestj = v, j
            cur[mid] = best
            solve(lo, mid - 1, optlo, bestj)      # left half: opt <= bestj
            solve(mid + 1, hi, bestj, opthi)      # right half: opt >= bestj
        solve(g - 1, n - 1, g - 2, n - 2)         # need >= g items to form g groups
        dp = cur
    return dp[n - 1]

a = [4, 1, 3, 2, 5, 1, 2]
print("naive:", dpo_dc_naive(a, 3))
print("d&c  :", dpo_dc_fast(a, 3))

naive: 114
d&c  : 114


**Proof it's the same DP.** 400 random `(a, K)` instances with `1 ≤ K ≤ n`, asserting the divide-and-conquer layer fill equals the brute O(kn²) scan, then a timing run showing the `n² → n log n` per-layer drop.

In [7]:
def gen_dc():
    n = random.randint(1, 30)
    K = random.randint(1, n)
    a = [random.randint(0, 9) for _ in range(n)]
    return (a, K)

t = stress(dpo_dc_naive, dpo_dc_fast, gen_dc)
print(f"asserted equal on {t} random inputs")

N, K = 800, 30
big = [random.randint(0, 9) for _ in range(N)]
s = time.perf_counter(); A = dpo_dc_naive(big, K); t_naive = time.perf_counter() - s
s = time.perf_counter(); B = dpo_dc_fast(big, K);  t_fast  = time.perf_counter() - s
print(f"n={N} k={K}:  naive {t_naive*1e3:7.1f} ms   d&c {t_fast*1e3:6.1f} ms"
      f"   speedup x{t_naive/t_fast:.1f}   (match={A == B})")

asserted equal on 400 random inputs


n=800 k=30:  naive   804.5 ms   d&c   21.0 ms   speedup x38.3   (match=True)


## 5. Knuth's optimization — O(n³) → O(n²)

The interval-DP heavyweight. Cost of merging a range is the range itself, split anywhere:

> `dp[i][j] = min_{i ≤ k < j} ( dp[i][k] + dp[k+1][j] ) + W(i, j)`,  with `dp[i][i] = 0`.

(The **merge-stones** / optimal-file-merge problem: repeatedly merge adjacent piles, each merge costing the combined size; minimize the total.) There are O(n²) intervals and an O(n) split scan each → **O(n³)**.

When `W` satisfies the **quadrangle inequality** *and* is monotone on intervals, the optimal split point `opt[i][j]` is sandwiched:

> `opt[i][j-1] ≤ opt[i][j] ≤ opt[i+1][j]`.

So instead of scanning all of `[i, j)`, each interval only scans between its two neighbors' optima. Summed over a fixed interval length the bounds **telescope**, making each diagonal O(n) and the whole table **O(n²)**.

In [8]:
def dpo_knuth_naive(w):
    n = len(w)
    if n == 0:
        return 0
    pre = _prefix(w)
    def rng(i, j):
        return pre[j + 1] - pre[i]        # W(i, j) = sum of the range
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n + 1):
        for i in range(0, n - length + 1):
            j = i + length - 1
            best = min(dp[i][k] + dp[k + 1][j] for k in range(i, j))   # O(n) split scan
            dp[i][j] = best + rng(i, j)
    return dp[0][n - 1]

def dpo_knuth_fast(w):
    n = len(w)
    if n == 0:
        return 0
    pre = _prefix(w)
    def rng(i, j):
        return pre[j + 1] - pre[i]
    INF = float("inf")
    dp  = [[0] * n for _ in range(n)]
    opt = [[i for _ in range(n)] for i in range(n)]   # opt[i][i] = i
    for length in range(2, n + 1):
        for i in range(0, n - length + 1):
            j = i + length - 1
            best, bk = INF, i
            klo = max(i,     opt[i][j - 1])           # opt[i][j-1] <= opt[i][j]
            khi = min(j - 1, opt[i + 1][j])           # opt[i][j]   <= opt[i+1][j]
            for k in range(klo, khi + 1):             # narrowed scan
                v = dp[i][k] + dp[k + 1][j]
                if v < best:
                    best, bk = v, k
            dp[i][j]  = best + rng(i, j)
            opt[i][j] = bk
    return dp[0][n - 1]

w = [4, 1, 3, 2, 6]
print("naive:", dpo_knuth_naive(w))
print("knuth:", dpo_knuth_fast(w))

naive: 36
knuth: 36


**Proof it's the same DP.** 400 random pile-weight arrays asserting Knuth's narrowed scan equals the full O(n³) split search, then timing the O(n³) → O(n²) collapse at `n = 300`.

In [9]:
def gen_knuth():
    n = random.randint(1, 40)
    w = [random.randint(1, 20) for _ in range(n)]
    return (w,)

t = stress(dpo_knuth_naive, dpo_knuth_fast, gen_knuth)
print(f"asserted equal on {t} random inputs")

N = 300
big = [random.randint(1, 20) for _ in range(N)]
s = time.perf_counter(); A = dpo_knuth_naive(big); t_naive = time.perf_counter() - s
s = time.perf_counter(); B = dpo_knuth_fast(big);  t_fast  = time.perf_counter() - s
print(f"n={N}:  naive {t_naive*1e3:7.1f} ms   knuth {t_fast*1e3:6.1f} ms"
      f"   speedup x{t_naive/t_fast:.1f}   (match={A == B})")

asserted equal on 400 random inputs


n=300:  naive   245.4 ms   knuth   16.5 ms   speedup x14.9   (match=True)


## 6. Which optimization fits which transition + cheat-sheet

The recognition table — match the **shape of the recurrence** to the accelerator:

| Recurrence shape | Trigger to look for | Optimization | Container / idea |
|---|---|---|---|
| `dp[i] = cost[i] + min(dp[j] : j ∈ [i-k, i-1])` | min/max over a **fixed window** of states | sliding-window DP | monotonic **deque** |
| `dp[i] = base[i] + min_j (m_j·x_i + b_j)` | transition **linear in a query value** `x_i` | **Convex Hull Trick** | lower envelope of lines |
| `dp[g][i] = min_j dp[g-1][j] + C(j,i)` | layered split, `opt[i]` **monotone in `i`** | **divide-and-conquer DP** | recurse on the argmin |
| `dp[i][j] = min_k dp[i][k]+dp[k+1][j] + W(i,j)` | interval cost with the **quadrangle inequality** | **Knuth's optimization** | `opt[i][j-1] ≤ opt[i][j] ≤ opt[i+1][j]` |

**Cheat-sheet**

| Optimization | From | To | Precondition (the structural property) |
|---|:---:|:---:|---|
| Sliding-window deque | O(nk) | **O(n)** | transition is a min/max over a fixed-width window |
| Convex Hull Trick (monotone) | O(n²) | **O(n)** | linear transition; slopes monotone, queries monotone |
| Convex Hull Trick (general) | O(n²) | **O(n log n)** | linear transition; arbitrary slopes/queries (sorted hull + binary search) |
| Divide-and-conquer DP | O(kn²) | **O(kn log n)** | `opt(i)` non-decreasing in `i` (implied by the QI) |
| Knuth's optimization | O(n³) | **O(n²)** | interval `W` satisfies the quadrangle inequality + monotonicity |

**Three things to remember.**

- **The recurrence is already correct — you're only changing *how fast* you evaluate it.** That's why every routine above is asserted equal to its naive DP: an optimization that changes the answer is just a bug.
- **The deck of conditions overlaps.** The quadrangle inequality is the common engine: it makes `opt` monotone (powering both D&C DP and Knuth) and it's what makes the relevant cost a lower-convex family (powering the hull trick). Spot a convex/QI cost and you usually have a choice of tool.
- **Match the dimension to the cost.** A 1-D `min` over earlier states → deque or hull; a 2-D layered split → divide-and-conquer; a 2-D interval merge → Knuth. These are the workhorses behind **advanced dynamic programming**; the discard-what-can't-win deque is the same instinct as **monotonic stack & queue**, now driving a DP instead of a scan.